Image to Text

In [1]:
import base64
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
load_dotenv()

True

In [2]:
image_path='image1.png'

In [3]:
llm=ChatOpenAI(
    model='gpt-5.4-mini'
)

In [4]:
def get_image(image_path):
    with open(image_path,'rb') as image:
        feed_text=base64.b64encode(image.read()).decode('utf-8').strip()
        return f"data:image/png;base64,{feed_text}"

In [5]:
value=get_image(image_path)

In [6]:
send_msg=HumanMessage(
    content=[
        {'type':'text','text':'Describe this given image perfectly and should not contain ** and ###'},
        {'type':'image_url','image_url':{'url':f'{value}'}}
    ]
)

In [7]:
result=llm.invoke([send_msg])

In [8]:
result

AIMessage(content='The image shows a serene forested landscape in autumn, viewed from an elevated vantage point. A calm, dark blue-green river curves diagonally through the lower half of the scene, bordered by a narrow rocky bank on one side and dense vegetation on both sides. The surrounding hills are covered with thick mixed woodland, with trees displaying a rich range of fall colors including deep green, golden yellow, orange, and muted brown. Tall conifers stand among deciduous trees, adding texture and variation to the canopy. The overall mood is quiet, natural, and peaceful, with no people, buildings, or man-made objects visible.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 2922, 'total_tokens': 3049, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache

In [9]:
print(result.content)

The image shows a serene forested landscape in autumn, viewed from an elevated vantage point. A calm, dark blue-green river curves diagonally through the lower half of the scene, bordered by a narrow rocky bank on one side and dense vegetation on both sides. The surrounding hills are covered with thick mixed woodland, with trees displaying a rich range of fall colors including deep green, golden yellow, orange, and muted brown. Tall conifers stand among deciduous trees, adding texture and variation to the canopy. The overall mood is quiet, natural, and peaceful, with no people, buildings, or man-made objects visible.


In [10]:
result.usage_metadata

{'input_tokens': 2922,
 'output_tokens': 127,
 'total_tokens': 3049,
 'input_token_details': {'audio': 0, 'cache_read': 2304},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [11]:
total_token=result.usage_metadata['total_tokens']

In [12]:
total_token

3049

Text to Image

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from openai import OpenAI

In [26]:
user_query="A high-resolution photograph of a serene glass mirror resting vertically on an asphalt road in a dry, sun-baked desert. The mirror reflects a lush, rain-drenched tropical rainforest under a stormy gray sky. However, in the non-reflected world outside the mirror, the surrounding desert is completely dry, hot, and sunny. Standing right at the edge of the mirror is a lone traveller wearing a yellow rain poncho; their reflection in the mirror shows them wearing a dry linen shirt. Perfect physics-compliant shadows, photorealistic details."

In [15]:
llm=ChatOpenAI(model='gpt-5.4-mini')

In [27]:
prompt = PromptTemplate(
    template="""You are an expert prompt engineer specializing in high-end AI image generation models (such as DALL-E 3 and OpenAI's GPT image models).

Your task is to transform the user's brief query into a rich, detailed, and visually stunning prompt.

User Query: {user_query}

Guidelines for the Expanded Prompt:
1. **Fill in the Gaps:** If the user query is brief or ambiguous, creatively add specific details (e.g., precise subjects, background elements, textures) to remove any guesswork.
2. **Visual Direction:** Explicitly define artistic style, lighting (e.g., golden hour, studio softbox), camera angle or composition (e.g., macro shot, wide-angle cinematic), and color palette.
3. **Clarity & Precision:** Eliminate potential hallucinations or weird text artifacts by keeping descriptions concrete and descriptive.
4. **Output Constraint:** Return *only* the final expanded image prompt text without any extra conversational filler or quotation marks.

Expanded Image Prompt:""",
    input_variables=["user_query"],
)

In [28]:
chain =prompt | llm | StrOutputParser()

result=chain.invoke({'user_query':user_query})

In [29]:
client=OpenAI()

In [30]:
response=client.images.generate(
    model='gpt-image-1-mini',
    prompt=result,
    size='1024x1024',
    quality='high',
    n=2
)

In [31]:
import json

print(json.dumps(response.model_dump(), indent=2))


{
  "created": 1792408871,
  "background": "opaque",
  "data": [
    {
      "b64_json": "...",
      "revised_prompt": null,
      "url": null,
      "generation_id": "f206b801-18b7-4e7e-8ac0-cd734e99be52"
    },
    {
      "b64_json": "...",
      "revised_prompt": null,
      "url": null,
      "generation_id": "c40a098a-2feb-47ef-b5de-b20f6fbbafc5"
    }
  ],
  "output_format": "png",
  "quality": "high",
  "size": "1024x1024",
  "usage": {
    "input_tokens": 754,
    "input_tokens_details": {
      "image_tokens": 0,
      "text_tokens": 754
    },
    "output_tokens": 8320,
    "total_tokens": 9074,
    "output_tokens_details": {
      "image_tokens": 8320,
      "text_tokens": 0
    }
  },
  "safety_identifier": null
}


In [34]:
# Loop through all items in response.data
for index, item in enumerate(response.data):
    image_base64 = item.b64_json
    image_bytes = base64.b64decode(image_base64)
    
    # Save each image uniquely (e.g., tti-output_1.png and tti-output_2.png)
    filename = f"tti-output_{index + 1}.png"
    with open(filename, "wb") as f:
        f.write(image_bytes)
        
    print(f"Saved {filename}")

Saved tti-output_1.png
Saved tti-output_2.png
